In [ ]:
# Block 0: Fix lỗi phiên bản Protobuf
!pip install "protobuf<4" --force-reinstall

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import matplotlib.pyplot as plt
import numpy as np
import os

# Cấu hình đường dẫn dataset
# Lưu ý: Vì dataset gộp chung, ta sẽ trỏ thẳng vào thư mục chứa các sub-folders (class)
DATA_DIR = '/kaggle/input/alzheimers-multiclass-dataset-equal-and-augmented/combined_images'

# Các thông số kỹ thuật
IMG_SIZE = 128   # EfficientNetB0 chuẩn là 224, nhưng 128 vẫn chạy tốt và nhanh hơn
BATCH_SIZE = 32
RANDOM_SEED = 42

print(f"TensorFlow Version: {tf.__version__}")

In [ ]:
# 1. Load toàn bộ dataset
# image_dataset_from_directory sẽ tự tìm các sub-folder làm nhãn (NonDemented, etc.)
full_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    shuffle=True,           # Trộn ngẫu nhiên dữ liệu
    seed=RANDOM_SEED,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode='categorical' # Dùng one-hot encoding cho 4 class
)

class_names = full_ds.class_names
print(f"Các lớp (Classes): {class_names}")

# 2. Hàm chia dữ liệu thủ công
def get_dataset_partitions_tf(ds, train_split=0.7, val_split=0.15, test_split=0.15, shuffle=True, shuffle_size=10000):
    ds_size = len(ds) # Tổng số batch
    
    if shuffle:
        ds = ds.shuffle(shuffle_size, seed=RANDOM_SEED)
    
    train_size = int(train_split * ds_size)
    val_size = int(val_split * ds_size)
    
    # Cắt dữ liệu
    train_ds = ds.take(train_size)
    val_ds = ds.skip(train_size).take(val_size)
    test_ds = ds.skip(train_size).skip(val_size)
    
    return train_ds, val_ds, test_ds

train_ds, val_ds, test_ds = get_dataset_partitions_tf(full_ds)

print(f"Số batch Train: {len(train_ds)}")
print(f"Số batch Val: {len(val_ds)}")
print(f"Số batch Test: {len(test_ds)}")

# 3. Tối ưu hiệu năng (Caching & Prefetching)
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [ ]:
def build_model(num_classes):
    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    
    # Data Augmentation (Tăng cường dữ liệu nhẹ)
    x = layers.RandomFlip("horizontal")(inputs)
    x = layers.RandomRotation(0.1)(x)
    
    # Base Model: EfficientNetB0
    # include_top=False: Bỏ lớp output gốc
    # weights='imagenet': Sử dụng kiến thức đã học từ ImageNet
    base_model = EfficientNetB0(include_top=False, weights='imagenet', input_tensor=x)
    
    # QUAN TRỌNG: Đóng băng base_model, không cho cập nhật trọng số lúc đầu
    base_model.trainable = False 
    
    # Xây dựng phần đuôi (Classifier) mới
    x = base_model.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)  # Giảm overfitting
    
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    model = models.Model(inputs, outputs, name="Alzheimer_Model")
    return base_model, model # Trả về cả base_model để dùng cho bước Fine-tuning sau này

base_model, model = build_model(len(class_names))
model.summary()

In [ ]:
# Compile model
model.compile(
    optimizer=optimizers.Adam(learning_rate=0.001), # Learning rate cơ bản
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Train khoảng 10-15 epochs
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=12,
    callbacks=[EarlyStopping(patience=3, restore_best_weights=True)] # Dừng nếu không cải thiện
)

In [ ]:
print("\nBắt đầu Fine-tuning...")

# 1. Mở khóa toàn bộ base model
base_model.trainable = True

# 2. Compile lại model với Learning Rate RẤT NHỎ
model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-5), 
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# 3. Train tiếp
TOTAL_EPOCHS = 25 

history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    initial_epoch=history.epoch[-1], # Bắt đầu từ epoch cuối của giai đoạn 1
    epochs=TOTAL_EPOCHS,
    callbacks=[
        EarlyStopping(patience=4, restore_best_weights=True),
        ReduceLROnPlateau(factor=0.2, patience=2)
    ]
)

In [ ]:
# Nối lịch sử train của 2 giai đoạn để vẽ biểu đồ
acc = history.history['accuracy'] + history_fine.history['accuracy']
val_acc = history.history['val_accuracy'] + history_fine.history['val_accuracy']
loss = history.history['loss'] + history_fine.history['loss']
val_loss = history.history['val_loss'] + history_fine.history['val_loss']

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(acc, label='Training Accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.axvline(x=history.epoch[-1], color='green', linestyle='--', label='Start Fine-Tuning')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')

plt.subplot(1, 2, 2)
plt.plot(loss, label='Training Loss')
plt.plot(val_loss, label='Validation Loss')
plt.axvline(x=history.epoch[-1], color='green', linestyle='--', label='Start Fine-Tuning')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')
plt.show()

# Đánh giá trên tập Test
print("\nĐánh giá trên tập Test Set (Chưa từng thấy):")
test_loss, test_acc = model.evaluate(test_ds)
print(f"Test Accuracy: {test_acc*100:.2f}%")

In [ ]:
# Lưu toàn bộ model (kiến trúc + trọng số)
model.save('model.keras') 
print("Đã lưu model thành công!")